In [29]:
import numpy as np
import imageio
import seaborn as sns
import cv2


def mp4_to_numpy(video_path):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise IOError(f"Cannot open video file: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # Convert from BGR (OpenCV default) to RGB if needed
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return np.array(frames)  # Shape: (num_frames, height, width, 3)

optical_flows = np.load("opticalflow.npz")["optical_flows"]
images = mp4_to_numpy("videoDatasetUpsamplingOriginal.mp4")

of_y = optical_flows[..., 0]
of_x = optical_flows[..., 1]

print(f"raw images shape : {images.shape}")
print(f"optical flow x shape : {of_x.shape}")
print(f"optical flow y shape : {of_y.shape}")

raw images shape : (56, 192, 240, 3)
optical flow x shape : (55, 180, 240)
optical flow y shape : (55, 180, 240)


In [34]:
import numpy as np
import cv2
from skimage.metrics import structural_similarity as ssim

def warp_backward(prev_rgb, flow_yx):
    H, W = prev_rgb.shape[:2]
    dy = flow_yx[..., 0].astype(np.float32)
    dx = flow_yx[..., 1].astype(np.float32)

    X, Y = np.meshgrid(np.arange(W), np.arange(H))
    map_x = (X - dx).astype(np.float32)
    map_y = (Y - dy).astype(np.float32)

    pred = cv2.remap(prev_rgb, map_x, map_y, interpolation=cv2.INTER_LINEAR,
                     borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    valid = (map_x >= 0) & (map_x <= W - 1) & (map_y >= 0) & (map_y <= H - 1)
    return pred, valid

def psnr_from_mse(mse, data_range=255.0):
    if mse <= 0:
        return float('inf')
    return 20.0 * np.log10(data_range) - 10.0 * np.log10(mse)

def compute_metrics(gt, pred, valid_mask):
    gt_f   = gt.astype(np.float32)
    pred_f = pred.astype(np.float32)
    vm = valid_mask

    diff = (gt_f - pred_f)[vm]
    mae = float(np.mean(np.abs(diff)))
    mse = float(np.mean(diff**2))
    psnr = psnr_from_mse(mse, data_range=255.0)

    pred_for_ssim = pred_f.copy()
    pred_for_ssim[~vm] = gt_f[~vm]
    ssim_val = float(ssim(gt_f, pred_for_ssim, data_range=255.0, channel_axis=2))
    return {'MAE': mae, 'PSNR': psnr, 'SSIM': ssim_val}

def try_variant_over_sequence(images_c, flows, swap_xy=False, flip_sign=False, return_raw=False):
    """
    images_c: (T, Hf, Wf, 3) uint8
    flows:    (T-1, Hf, Wf, 2)
    """
    if swap_xy:
        flow2 = np.stack([flows[...,1], flows[...,0]], axis=-1)
    else:
        flow2 = flows
    if flip_sign:
        flow2 = -flow2

    warped = []
    maes, psnrs, ssims = [], [], []

    for t in range(flow2.shape[0]):
        I_t, I_tp1 = images_c[t], images_c[t+1]
        pred, valid = warp_backward(I_t, flow2[t])
        warped.append(pred)
        m = compute_metrics(I_tp1, pred, valid)
        maes.append(m['MAE']); psnrs.append(m['PSNR']); ssims.append(m['SSIM'])

    if not return_raw:
        return {'MAE': np.mean(maes), 'PSNR': np.mean(psnrs), 'SSIM': np.mean(ssims)}

    return {'MAE': np.mean(maes), 'PSNR': np.mean(psnrs), 'SSIM': np.mean(ssims),
                "raw" : warped}

def evaluate_with_padding_search(images_rgb, flows_yx):
    T, Himg, Wimg, _ = images_rgb.shape
    Tf, Hf, Wf, _ = flows_yx.shape
    best = None
    best_info = None

    for y0 in range(0, Himg - Hf + 1):
        for x0 in range(0, Wimg - Wf + 1):
            imgs_c = images_rgb[:, y0:y0+Hf, x0:x0+Wf, :]
            variants = {
                'dy_dx':         try_variant_over_sequence(imgs_c, flows_yx, swap_xy=False, flip_sign=False),
                'dy_dx_flipped': try_variant_over_sequence(imgs_c, flows_yx, swap_xy=False, flip_sign=True),
                'dx_dy':         try_variant_over_sequence(imgs_c, flows_yx, swap_xy=True,  flip_sign=False),
                'dx_dy_flipped': try_variant_over_sequence(imgs_c, flows_yx, swap_xy=True,  flip_sign=True),
            }

            pick = min(variants.items(), key=lambda kv: (kv[1]['MAE'], -kv[1]['PSNR'], -kv[1]['SSIM']))
            score = pick[1]
            if (best is None) or (score['MAE'] < best['MAE'] or
                                  (score['MAE'] == best['MAE'] and (score['PSNR'] > best['PSNR'] or
                                                                    (score['PSNR'] == best['PSNR'] and score['SSIM'] > best['SSIM'])))):
                best = score
                best_info = {'y0': y0, 'x0': x0, 'variant': pick[0], 'scores': score}

    return best_info

info = evaluate_with_padding_search(images, optical_flows)
print("Best crop/convention:")
print(f"  crop y0={info['y0']}, x0={info['x0']}  (size = {optical_flows.shape[1]}x{optical_flows.shape[2]})")
print(f"  variant = {info['variant']}")
print(f"  MAE={info['scores']['MAE']:.2f}  PSNR={info['scores']['PSNR']:.2f}  SSIM={info['scores']['SSIM']:.4f}")

Best crop/convention:
  crop y0=12, x0=0  (size = 180x240)
  variant = dx_dy
  MAE=2.72  PSNR=30.65  SSIM=0.9611


In [40]:
ret = try_variant_over_sequence(images[:, 12:], optical_flows, swap_xy=True, return_raw=True)
raw = ret['raw']

print("MAE : ", ret["MAE"])
print("PSNR : ", ret["PSNR"])
print("SSIM : ", ret["SSIM"])

MAE :  2.7157105619257145
PSNR :  30.6542242118138
SSIM :  0.9610995249314742


In [47]:
import matplotlib.pyplot as plt
import imageio

true_values = images[1:]

with imageio.get_writer("stiched.mp4", fps=10) as writer:
    for i in range(len(raw)):
       stiched = np.concatenate([raw[i], true_values[i, 12:]], axis=1)
       writer.append_data(stiched)


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (480, 180) to (480, 192) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


# Get from dataset

In [63]:
import numpy as np
import imageio
import seaborn as sns
import cv2
import json

def mp4_to_numpy(video_path):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise IOError(f"Cannot open video file: {video_path}")

    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # Convert from BGR (OpenCV default) to RGB if needed
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return np.array(frames)  # Shape: (num_frames, height, width, 3)

optical_flows = np.load("optical_flow.npz")['optical_flow']
images = mp4_to_numpy("video.mp4")

with open("params.json", "r") as f:
    params = json.load(f)

deltas = params['images']

raw_images = []
prev = 0
for delta in deltas:
    raw_images.append(images[prev])
    prev += delta

raw_images.append(images[-1])
raw_images = np.array(raw_images)

In [66]:
ret = try_variant_over_sequence(raw_images, optical_flows, swap_xy=True)
print(ret)

{'MAE': np.float64(2.988931924646551), 'PSNR': np.float64(30.362471528737647), 'SSIM': np.float64(0.9554326740178195)}
